# GT4Py Hands-on

This notebook will guide you towards a fully (or at least partially) GT4Py-based implementation of the `stencil2d.py` program we saw on Day 1. The module `stencil2d-gt4py-v0.py` contains the backbone of the final code. Holes which need to be filled in with your inputs are marked as `# TODO`. Here we go through the porting process step-by-step. You will have the opportunity to implement all missing parts in isolation and test them standalone. Once you complete all the mandatory tasks successfully, you can copy and paste the relevant cells of this notebook into `stencil2d-gt4py-v0.py` to have a running GT4Py program. To keep our lives simple, we shall confine our attention to two CPU execution modes: `None` (embedded Python execution using NumPy arrays) and `gtx.gtfn_cpu` (compiled CPU execution).

In [ ]:
from typing import Callable

import gt4py.next as gtx
import numpy as np

I = gtx.Dimension("I")
J = gtx.Dimension("J")
K = gtx.Dimension("K")

IJKField = gtx.Field[gtx.Dims[I, J, K], gtx.float64]
OFFSET_PROVIDER = {"_IOff": I, "_JOff": J}

## Stencil computations

Let's start by implementing the 5-points Laplacian stencil 
```
lap_field[i, j, k] = - 4 * in_field[  i,   j, k] 
                     +     in_field[i-1,   j, k] 
                     +     in_field[i+1,   j, k] 
                     +     in_field[  i, j-1, k] 
                     +     in_field[  i, j+1, k]
```
as a field operator.

<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
<b>1.</b> Fill the GT4Py field operator <tt>laplacian</tt> whose signature is already provided. <br>
</div>

In [2]:
@gtx.field_operator
def laplacian(in_field: IJKField) -> IJKField:
    lap_field = (
        -4.0 * in_field + in_field(I - 1) + in_field(I + 1) + in_field(J - 1) + in_field(J + 1)
    )
    return lap_field

We now introduce another level of abstraction with respect to `stencil2d.py`. Leveraging the `laplacian` subroutine we implement a stencil which applies the fourth-order diffusion operator

\begin{equation}
    \frac{\partial \phi}{\partial t} = - \alpha_4 \, \Delta_h \, (\Delta_h \phi) \, .
\end{equation}

<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
<b>2.</b> Write a field operator called <tt>diffusion</tt> which performs the same operations as the following lines in <tt>stencil2d.py</tt>:<br>
<code>laplacian( in_field, tmp_field, num_halo=num_halo, extend=1 )
 laplacian( tmp_field, out_field, num_halo=num_halo, extend=0 )
 out_field[:, num_halo:-num_halo, num_halo:-num_halo] = \
     in_field[:, num_halo:-num_halo, num_halo:-num_halo] \
     - alpha * out_field[:, num_halo:-num_halo, num_halo:-num_halo] </code><br>
The function receives the input field <tt>in_field</tt>, the output field <tt>out_field</tt> and the scalar coefficient <tt>alpha</tt>. Assume grid point values are stored as <tt>float</tt>s. Import and call the <tt>laplacian</tt> subroutine.<br>
<b>3.</b> Instantiate the field operator with the two different backends.
</div>

In [ ]:
@gtx.field_operator
def diffusion(in_field: IJKField, alpha: gtx.float64) -> IJKField:
    lap1 = laplacian(in_field)
    lap2 = laplacian(lap1)
    return in_field - alpha * lap2


@gtx.program
def diffusion_program(
    in_field: IJKField,
    out_field: IJKField,
    alpha: gtx.float64,
    nx: gtx.int32,
    ny: gtx.int32,
    nz: gtx.int32,
):
    diffusion(in_field, alpha, out=out_field, domain={I: (0, nx), J: (0, ny), K: (0, nz)})

In [ ]:
diffusion_stencil_embedded = diffusion_program.with_backend(None)

In [ ]:
diffusion_stencil_cpu = diffusion_program.with_backend(gtx.gtfn_cpu)

## Updating the boundary region

For GPU execution, updating halo values through the underlying array would require explicit host/device synchronization and is not the right model. A GT4Py implementation of the boundary conditions would be preferable, but also more cumbersome. Here we restrict ourselves to the CPU-oriented Python halo update.

<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
<b>4.</b> The <tt>update_halo()</tt> function receives (i) the GT4Py field on which periodicity should be imposed and (ii) the width of the halo. Write the body of the function by indexing <tt>field.ndarray</tt> as a regular <tt>numpy.ndarray</tt>. Recall that the axes order is <tt>I-J-K</tt>, while in <tt>stencil2d.py</tt> we adopted the Fortran-ish <tt>K-J-I</tt> order. Validate your code using <tt>test_update_halo()</tt>. <br>
</div>

In [ ]:
def update_halo(field: IJKField, num_halo: int):
    # Make sure to use field.ndarray here

    # bottom edge (without corners)
    field.ndarray[num_halo:-num_halo, :num_halo] = field.ndarray[
        num_halo:-num_halo, -2 * num_halo : -num_halo
    ]

    # top edge (without corners)
    field.ndarray[num_halo:-num_halo, -num_halo:] = field.ndarray[
        num_halo:-num_halo, num_halo : 2 * num_halo
    ]

    # left edge (including corners)
    field.ndarray[:num_halo, :] = field.ndarray[-2 * num_halo : -num_halo, :]

    # right edge (including corners)
    field.ndarray[-num_halo:, :] = field.ndarray[num_halo : 2 * num_halo]

In [ ]:
def test_update_halo(f):
    data = np.load("baseline_data/update_halo.npz")
    in_array = data["in_field"].copy()
    val = data["out_field"]
    num_halo = data["num_halo"]
    domain = {I: in_array.shape[0], J: in_array.shape[1], K: in_array.shape[2]}
    field = gtx.as_field(domain, in_array)

    f(field, num_halo)

    if np.allclose(field.asnumpy(), val):
        print("Unit test for update_halo(): PASSED!")
    else:
        print("Unit test for update_halo(): FAILED.")


test_update_halo(update_halo)

## Time integration

The time marching procedure is carried out by the `apply_diffusion` function, whose signature reads:

```
def apply_diffusion(diffusion_stencil, in_field, out_field, alpha, num_halo, num_iter=1):
```

Here `diffusion_stencil` is the stencil object which applies the diffusion operator, `in_field` and `out_field` are the input and output fields, `alpha` is the diffusion coefficient, `num_halo` is the number of halo points, and `num_iter` is the number of iterations. Each iteration consists of three steps:

1. Updating the halo region of the input field `in_field`;
2. Running the `diffusion` stencil on `in_field` and store the results in `out_field`;
3. Updating the halo region of the output field `out_field` if it is the last iteration, otherwise swapping `in_field` and `out_field`.

<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
    <b>5.</b> Determine the interior extent of the computation based on <tt>num_halo</tt> and the size of the input field. Hint: use the <tt>shape</tt> attribute of a GT4Py field to retrieve its size.<br>
    <b>6.</b> Add the call to <tt>diffusion_stencil</tt>. <br>
</div>


In [ ]:
def apply_diffusion(
    diffusion_stencil: Callable,
    in_field: IJKField,
    out_field: IJKField,
    alpha: gtx.float64,
    num_halo: int,
    num_iter: int = 1,
):
    nx = in_field.shape[0] - 2 * num_halo
    ny = in_field.shape[1] - 2 * num_halo
    nz = in_field.shape[2]

    for n in range(num_iter):
        # halo update
        update_halo(in_field, num_halo)

        # run the stencil
        diffusion_stencil(
            in_field,
            out_field,
            alpha,
            nx,
            ny,
            nz,
            offset_provider=OFFSET_PROVIDER,
        )

        if n < num_iter - 1:
            # swap input and output fields
            in_field, out_field = out_field, in_field
        else:
            # halo update
            update_halo(out_field, num_halo)

## Input and output fields

We are almost done. The last mile concerns the fields which contain the input and output data. We explained in `02-GT4Py-concepts.ipynb` that GT4Py programs operate on `gtx.Field` objects. Fields can be allocated with GT4Py helpers such as `gtx.zeros` or created from existing NumPy/CuPy arrays with `gtx.as_field`. The domain describes the logical index space, while the allocator/backend controls the memory layout used for the target architecture. This keeps the user interface hardware-agnostic.

<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
    <b>7.</b> Based on what we said in <tt>02-GT4Py-concepts.ipynb</tt>, what is the appropriate domain for the input field? Assume <tt>num_halo = 2</tt>.<br>
    <b>8.</b> Define a GT4Py field called <tt>in_field</tt> and fill it with zeros. Make use of the <tt>backend</tt> variable to set the backend. Then use NumPy to fill in some actual data.<br>
    <b>9.</b> Allocate an empty GT4Py field <tt>out_field</tt> to hold the output data. <tt>out_field</tt> must have the same shape as <tt>in_field</tt>. Pick the same backend as for <tt>in_field</tt>.<br>
    <b>10.</b> Use the just allocated fields to test <tt>diffusion_cpu</tt> via <tt>test_diffusion()</tt>.
</div>

In [9]:
nx = 128
ny = 128
nz = 64
num_halo = 2

# backend
backend = gtx.gtfn_cpu

# define domain
field_domain = {
    I: (-num_halo, nx + num_halo),
    J: (-num_halo, ny + num_halo),
    K: (0, nz),
}

# allocate input and output fields
in_field = gtx.zeros(field_domain, dtype=gtx.float64, allocator=backend)
out_field = gtx.zeros(field_domain, dtype=gtx.float64, allocator=backend)

# prepare input field
in_field[
    num_halo + nx // 4 : num_halo + 3 * nx // 4,
    num_halo + ny // 4 : num_halo + 3 * ny // 4,
    nz // 4 : 3 * nz // 4,
] = 1.0

In [ ]:
def test_diffusion(stencil_object, in_field, out_field, num_halo=2):
    phi = in_field.asnumpy()
    tmp1 = np.zeros_like(phi)
    tmp2 = np.zeros_like(phi)
    out = np.zeros_like(phi)
    interior = np.s_[num_halo:-num_halo, num_halo:-num_halo, :]
    nx = in_field.shape[0] - 2 * num_halo
    ny = in_field.shape[1] - 2 * num_halo
    nz = in_field.shape[2]

    tmp1[1:-1, 1:-1] = (
        phi[2:, 1:-1] + phi[:-2, 1:-1] + phi[1:-1, 2:] + phi[1:-1, :-2] - 4.0 * phi[1:-1, 1:-1]
    )
    tmp2[interior] = (
        tmp1[3:-1, 2:-2]
        + tmp1[1:-3, 2:-2]
        + tmp1[2:-2, 3:-1]
        + tmp1[2:-2, 1:-3]
        - 4.0 * tmp1[interior]
    )
    out[interior] = phi[interior] - 2.0 * tmp2[interior]

    stencil_object(
        in_field,
        out_field,
        2.0,
        nx,
        ny,
        nz,
        offset_provider=OFFSET_PROVIDER,
    )

    backend_name = "embedded" if stencil_object.backend is None else stencil_object.backend.name
    if np.allclose(out_field.asnumpy()[interior], out[interior]):
        print(f"Unit test for diffusion_{backend_name}: PASSED!")
    else:
        print(f"Unit test for diffusion_{backend_name}: FAILED.")

In [11]:
test_diffusion(diffusion_stencil_cpu, in_field, out_field)

Unit test for diffusion_run_gtfn_cpu_cached: PASSED!


## Running

All right! We are now ready to move onto `stencil2d-gt4py-v0.py`.

<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
    <b>11.</b> Take some time to understand the structure of the code and realize which parts of this notebook can be transferred as they are (or upon little modifications) into the script.<br>
    <b>12.</b> Fill the holes marked with <tt># TODO</tt> by copy and paste from this notebook.
</div>

Let's run `stencil2d-gt4py-v0.py` and check that the stencil runs fine with both backends:

In [12]:
%%bash
python stencil2d-gt4py-v0.py --nx=32 --ny=32 --nz=32 --num_iter=1024 --backend=None

Elapsed time for work = 1.52775239944458 s


In [14]:
%%bash
python stencil2d-gt4py-v0.py --nx=32 --ny=32 --nz=32 --num_iter=1024 --backend=cpu

Elapsed time for work = 5.271944522857666 s


<div class="alert alert-block alert-info">
<b>Now it's your turn...</b><br>
    <b>13.</b> From a terminal, execute the <tt>validation.sh</tt> Bash script to validate the numerics of your code. This script takes two command line arguments: the version tag of the program (here <code>v0</code>) and the desired backend. <br>
</div>

<div class="alert alert-block alert-success">
<i>All Bonus questions can be run directly from a terminal.</i><br>
<b>14. (Bonus)</b> Run both <tt>stencil2d.py</tt> and <tt>stencil2d-gt4py-v0.py</tt> with <code>--nx=128 --ny=128 --nz=64 --num_iter=1024</code>. How does the performance of the different CPU backends of GT4Py compare? <br>
<b>15. (Bonus)</b> Increase the grid size to <code>nx x ny x nz = 256 x 128 x 64</code> and then <code>nx x ny x nz = 256 x 256 x 64</code>. Speculate how the speed-up provided by the <tt>gtx.gtfn_cpu</tt> backend varies with the number of grid points.
</div>

*Solution discussion:*

On Santis with `prgenv-gnu/26.3:v1`, GT4Py 1.1.10, and 72 CPU cores, the median elapsed times for `--num_iter=1024` were:

| grid size | `stencil2d.py` | `stencil2d-gt4py-v0.py --backend=None` | `stencil2d-gt4py-v0.py --backend=cpu` |
|---|---:|---:|---:|
| `128 x 128 x 64` | 6.63 s | 9.76 s | 0.780 s |
| `256 x 128 x 64` | 13.40 s | 18.58 s | 0.783 s |
| `256 x 256 x 64` | 48.03 s | 58.36 s | 1.138 s |

The `None` backend is useful for debugging and correctness checks, but it is not a performance backend. In this measurement it is slower than the NumPy reference. The compiled `gtx.gtfn_cpu` backend is the relevant CPU performance backend: it is about `8.5x`, `17x`, and `42x` faster than the NumPy version for these three grid sizes. The speed-up increases with the problem size because the generated compiled code amortizes fixed overheads and avoids much of the Python and temporary-array overhead in the NumPy implementation.

## Further optimizations

Let's try to apply a couple of optimizations to our code. We shall proceed along the lines of what we did in day 1 on `stencil2d.F90`. All the following tasks are optional and involve the `gtx.gtfn_cpu` backend only.

<div class="alert alert-block alert-success">
<b>16. (Bonus)</b> Make a copy of <tt>stencil2d-gt4py-v0.py</tt> and name it <tt>stencil2d-gt4py-v1.py</tt>. Inside <tt>diffusion</tt> inline the field operator <tt>laplacian</tt> by replacing the calls to the function with its body. Use the <tt>validation.sh</tt> script to validate your code. What is the performance gain with respect to <tt>stencil2d-gt4py-v0.py</tt>? Base your answer on the timings measured at different grid sizes. <br>
<b>17. (Bonus)</b> Here we go for a more aggressive optimization. Make a copy of <tt>stencil2d-gt4py-v1.py</tt> and name it <tt>stencil2d-gt4py-v2.py</tt>. Fuse all stages (i.e. statements) inside <tt>diffusion</tt> into a single stage, as done in <code>day1/stencil2d-inlining_v2.F90</code>. Modify the function signature by replacing <tt>alpha</tt> with <tt>a1 = - alpha</tt>, <tt>a2 = - 2 * alpha</tt>, <tt>a8 = 8 * alpha</tt> and <tt>a20 = 1 - 20 * alpha</tt>. Adapt the stencil call inside <tt>apply_diffusion</tt> accordingly. Validate your code using <tt>validation.sh</tt>. Can you observe any meaningful performance improvement? In your opinion, is the stencil code still understandable and intuitive as in <tt>stencil2d-gt4py-v0.py</tt>? <br>
</div>

*Solution discussion:*

Using the same setup, the median times for the compiled GT4Py CPU versions were:

| grid size | v0 | v1, inlined `laplacian` | v2, fully fused expression |
|---|---:|---:|---:|
| `128 x 128 x 64` | 0.780 s | 0.753 s | 0.808 s |
| `256 x 128 x 64` | 0.783 s | 0.788 s | 0.841 s |
| `256 x 256 x 64` | 1.138 s | 1.127 s | 1.218 s |

Inlining `laplacian` manually in v1 does not provide a meaningful performance improvement over v0; the timings are essentially identical. This suggests that GT4Py already has enough information from the composed field operators to generate efficient code. The more aggressive v2 formulation is consistently slower in this run and is also harder to read. For this stencil, the clearer v0/v1 structure is preferable; fully expanding the expression removes useful structure without improving performance.